<div style="border: solid green 2px; padding: 20px">

  <b>Overall Summary of the Project – Iteration 1</b><br><br>
  Hello Samiul, congratulations on submitting your project!<br>

  My name is <b>Victor Camargo</b> 
  (<a href="https://hub.tripleten.com/u/834cb557" target="_blank">TripleTen Hub profile</a>) and I’ll be reviewing your project today.<br>

  <i>You can find my detailed feedback throughout the notebook, starting with comments labeled 
  <b>"Reviewer’s comment – Iteration 1"</b>.</i><br>

  <b>What you did well:</b><br>
  ✅ You inspected the data thoroughly, used a stratified train validation test split, ran validation based hyperparameter searches, selected a Random Forest based on validation performance, and confirmed the final model beats a naive baseline while meeting the accuracy target.<br>

  <b>Revision items (only if red issues exist):</b><br>
  ⛔️ None. I did not find unresolved critical issues that block approval.

  <b>Project Status:</b><br>
  <div class="alert alert-success" style="border-left: 7px solid green; padding: 5px">
    <b>Approved</b>
  </div>

  <hr><b>Legend:</b><br>

  <div class="alert alert-success" style="border-left: 7px solid green; padding: 5px">
  <b>✅ Reviewer’s comment – Iteration 1:</b><br>
  Strong, correct solutions or good practices worth reusing.
  </div>

  <div class="alert alert-warning" style="border-left: 7px solid gold; padding: 5px">
  <b>⚠️ Reviewer’s comment – Iteration 1:</b><br>
  Recommended improvements to strengthen your work.
  </div>

  <div class="alert alert-danger" style="border-left: 7px solid red; padding: 5px">
  <b>⛔️ Reviewer’s comment – Iteration 1:</b><br>
  Required revisions to address in the next pass.
  </div>

  <div class="alert alert-info" style="border-left: 7px solid blue; padding: 5px">
  <b>Student’s Comment</b><br>
  You may add your own notes or explanations in a <b>Markdown cell</b> using:<br>
  <code>&lt;div class="alert alert-info" style="border-left: 7px solid blue"&gt;&lt;b&gt;Student’s Comment&lt;/b&gt;&lt;/div&gt;</code>
  </div>

  <hr>
  <b>Please ensure</b> all cells run smoothly from top to bottom and display their outputs.<br>
  <b>Kind reminder:</b> please do not remove or modify reviewer comments, as they help track progress.<br>
  If you have any questions or need clarification, feel free to use the <b>Questions</b> channel.

</div>

## 1. Open and look through the data

In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

In [2]:
os.getcwd()
os.listdir('.')
os.makedirs('datasets', exist_ok=True)

In [3]:
os.listdir('datasets')

['users_behavior.csv']

In [4]:
df = pd.read_csv('datasets/users_behavior.csv')
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [6]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [7]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows: ", df.duplicated().sum())

Missing values per column:
calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

Duplicate rows:  0


<div class="alert alert-success" style="border-left: 7px solid green; padding: 5px">
  <b>✅ Reviewer’s comment – Iteration 1:</b><br>
  Good work loading and exploring the data. You checked shapes, data types, missing values, duplicates, and descriptive statistics, and you explicitly inspected class balance. Calling out the 69 31 class ratio up front is helpful for interpreting accuracy later. These checks give a solid foundation for modeling.
</div>

In [8]:
df['is_ultra'].value_counts(normalize = True)

0    0.693528
1    0.306472
Name: is_ultra, dtype: float64

About 69% of users are on Smart (0) and 31% on Ultra (1). The classes are imbalanced but not severely so. This matters later: a model that always predicts "Smart" would already be right 69% of the time, so we need our real model to clearly beat that baseline, not just clear 0.75 by accident. We'll check this explicitly in the sanity-check section.

## 2. Split the data into train, validation, and test sets

In [9]:
features = df.drop(['is_ultra'], axis = 1)
target = df['is_ultra']

# First split: 60% train, 40% temp
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size = 0.4, random_state = 42, stratify = target)

# Second split: split temp 50/50 into validation and test (20% / 20% of original)
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=42, stratify=target_temp)

print("Train set: ", features_train.shape)
print("Validation set: ", features_valid.shape)
print("Test set: ", features_test.shape)

Train set:  (1928, 4)
Validation set:  (643, 4)
Test set:  (643, 4)


<div class="alert alert-success" style="border-left: 7px solid green; padding: 5px">
  <b>✅ Reviewer’s comment – Iteration 1:</b><br>
  Nice stratified splitting into training validation and test sets with a fixed random state. Using stratify preserves class balance across sets and your printed shapes confirm the intended 60 20 20 proportion. The short justification you added is clear and appropriate.
</div>

The source file has no separate test set, so we need to carve one out ourselves. Since we don't have a hidden test set to validate against, a common and reasonable approach is a 3:1:1 split:

- 60% train — used to fit each candidate model
- 20% validation — used to compare hyperparameters and pick the best model
- 20% test — used only once, at the very end, to get an unbiased estimate of final quality

We do this in two steps with 'train_test_split': first split off 40% as a temporary set, then split that in half into validation and test. We use 'stratify=y' so all three sets preserve the same -69/31 class balance as the full dataset, and a fixed 'random_state' for reproducibility.

## 3. Investigate model quality with different hyperparameters

We'll try three model types and tune their key hyperparameters using the **validation set**:

- **Decision Tree** — vary 'max_depth'
- **Random Forest** — vary 'n_estimators' and 'max_depth'
- **Logistic Regression** — as a fast linear baseline (no hyperparameter sweep needed)

For each, we train on 'features_train' / 'target_train' and score on 'features_valid' / 'target_valid'.

## 3.1 Decision Tree

In [10]:
best_dt_model = None
best_dt_accuracy = 0
best_dt_depth = 0

for depth in range(1, 16):
    model = DecisionTreeClassifier(random_state = 42, max_depth = depth)
    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)
    accuracy = accuracy_score(target_valid, predictions)
    print(f"max_depth={depth:2d}  validation accuracy={accuracy:.4f}")
    if accuracy > best_dt_accuracy:
        best_dt_accuracy = accuracy
        best_dt_depth = depth
        best_dt_model = model


print()
print(f"Best Decision Tree: max_depth={best_dt_depth}, validation accuracy={best_dt_accuracy:.4f}")


max_depth= 1  validation accuracy=0.7605
max_depth= 2  validation accuracy=0.7869
max_depth= 3  validation accuracy=0.8040
max_depth= 4  validation accuracy=0.7963
max_depth= 5  validation accuracy=0.7900
max_depth= 6  validation accuracy=0.7745
max_depth= 7  validation accuracy=0.7885
max_depth= 8  validation accuracy=0.7792
max_depth= 9  validation accuracy=0.7807
max_depth=10  validation accuracy=0.7698
max_depth=11  validation accuracy=0.7760
max_depth=12  validation accuracy=0.7745
max_depth=13  validation accuracy=0.7807
max_depth=14  validation accuracy=0.7496
max_depth=15  validation accuracy=0.7465

Best Decision Tree: max_depth=3, validation accuracy=0.8040


Accuracy rises as the tree gets a bit deeper, then starts to fall — a classic sign of overfitting once the tree is allowed to grow too complex and starts memorizing the training data instead of generalizing.

### 3.2 Random Forest

In [11]:
best_rf_model = None
best_rf_accuracy = 0
best_rf_params = None

for est in [10, 20, 30, 40, 50]:
    for depth in [5, 6, 7, 8, 9, 10, None]:
        model = RandomForestClassifier(random_state = 42, n_estimators = est, max_depth = depth)
        model.fit(features_train, target_train)
        predictions = model.predict(features_valid)
        accuracy = accuracy_score(target_valid, predictions)
        if accuracy > best_rf_accuracy:
            best_rf_accuracy = accuracy
            best_rf_params = (est, depth)
            best_rf_model = model

print(f"Best Random Forest: n_estimators={best_rf_params[0]}, max_depth={best_rf_params[1]}")
print(f"Validation accuracy={best_rf_accuracy:.4f}")

Best Random Forest: n_estimators=50, max_depth=5
Validation accuracy=0.8118


<div class="alert alert-warning" style="border-left: 7px solid gold; padding: 5px">
  <b>⚠️ Reviewer’s comment – Iteration 1:</b><br>
  As an optional improvement, consider using cross validation or GridSearchCV or RandomizedSearchCV to make the hyperparameter search more robust to variance in the validation fold. This is not required, but it can increase confidence in the chosen parameters.
</div>

<div class="alert alert-success" style="border-left: 7px solid green; padding: 5px">
  <b>✅ Reviewer’s comment – Iteration 1:</b><br>
  Well done performing hyperparameter searches for both the Decision Tree and the Random Forest using the validation set. You compared multiple model families including a linear baseline and selected the Random Forest based on validation accuracy, which is a sound approach.
</div>

The Random Forest outperforms the single Decision Tree, which is expected — averaging predictions over many trees reduces overfitting and variance compared to a single tree.

### 3.3 Logistic Regression (baseline)

In [12]:
lr_model = LogisticRegression(random_state = 42, solver='liblinear')
lr_model.fit(features_train, target_train)
lr_predictions = lr_model.predict(features_valid)
lr_accuracy = accuracy_score(target_valid, lr_predictions)

print(f"Logistic Regression validation accuracy={lr_accuracy:.4f}")

Logistic Regression validation accuracy=0.7014


## Findings:
- The **Random Forest** achieved the highest validation accuracy, followed by the tuned **Decision Tree**, with **Logistic Regression** clearly the weakest of the three.
- This ordering makes sense: the relationship between usage behavior and plan choice is likely non-linear (e.g., there may be thresholds in 'minutes' or 'mb_used' where users switch plans), which tree-based models capture more naturally than a linear model.
- The Random Forest's advantage over the single Decision Tree comes from ensembling — averaging many trees smooths out the overfitting that hurt the single tree at larger depths.
- Trade-off to note: Random Forest is slower to train and to predict than a single Decision Tree or Logistic Regression, which could matter if this model needs to run on very large subscriber volumes or in real time. For this project's scale (a few thousand rows), that cost is negligible.
- We'll move forward with the Random Forest as our final model, since accuracy is the primary criterion here.

## 4. Check final model quality on the test set

In [13]:
final_model = RandomForestClassifier(random_state = 42, n_estimators = best_rf_params[0], max_depth = best_rf_params[1])
final_model.fit(features_train, target_train)
test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Meets 0.75 threshold: {test_accuracy >= 0.75}")

Test accuracy: 0.7932
Meets 0.75 threshold: True


Now that we've picked the Random Forest (with the best hyperparameters found above) using the validation set, we retrain it on the training set and evaluate it once on the held-out test set, which neither training nor hyperparameter selection has touched.

## 5. Sanity check

A model can look good on accuracy alone and still be doing something trivial — especially with class imbalance (69% Smart / 31% Ultra here). As a sanity check, we compare our model against a naive baseline that always predicts the majority class (Smart), using `DummyClassifier`.

If our model isn't meaningfully better than always guessing "Smart," it isn't actually learning useful behavior patterns — it would just be exploiting the class imbalance.

In [14]:
dummy_model = DummyClassifier(strategy='most_frequent', random_state = 42)
dummy_model.fit(features_train, target_train)
dummy_predictions = dummy_model.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f"Dummy baseline (always predict Smart) test accuracy: {dummy_accuracy:.4f}")
print(f"Our Random Forest test accuracy:                     {test_accuracy:.4f}")
print(f"Improvement over baseline:                            {test_accuracy - dummy_accuracy:.4f}")

importances = pd.Series(final_model.feature_importances_, index=features_train.columns)
importances.sort_values(ascending=False)

Dummy baseline (always predict Smart) test accuracy: 0.6936
Our Random Forest test accuracy:                     0.7932
Improvement over baseline:                            0.0995


mb_used     0.370062
minutes     0.244534
calls       0.228025
messages    0.157379
dtype: float64

Sanity check conclusion:
- Our Random Forest clearly outperforms the naive "always predict Smart" baseline, confirming it has learned real patterns in subscriber behavior rather than just exploiting class imbalance.
- The feature importances also make intuitive sense: usage-volume features like 'minutes' and 'mb_used' carry the most weight in predicting plan choice, which lines up with the idea that heavier users are more likely to be recommended the Ultra plan. This gives us confidence the model is behaving sensibly, not picking up on noise.

## Conclusion

We split the Megaline subscriber behavior data into training (60%), validation (20%), and test (20%) sets, then compared Decision Tree, Random Forest, and Logistic Regression models by tuning hyperparameters on the validation set. The Random Forest model performed best and, when evaluated on the untouched test set, achieved an accuracy above the required **0.75** threshold. A sanity check against a naive majority-class baseline and an inspection of feature importances both confirm the model is learning genuine, sensible patterns in usage behavior rather than exploiting class imbalance. This model is ready to be used to recommend the Smart or Ultra plan to legacy-plan subscribers.

<div class="alert alert-success" style="border-left: 7px solid green; padding: 5px">
  <b>✅ Reviewer’s comment – Iteration 1:</b><br>
  Good final evaluation and sanity checks. You retrained the chosen model using the selected hyperparameters and evaluated once on the held out test set, and you compared performance to a majority class baseline. The Random Forest test accuracy of 0.7932 meets the stated target and the improvement over the baseline shows the model learned meaningful patterns. Your final conclusion is concise and well stated.
</div>